# Практическая работа №4: Временные ряды и пользовательские сессии

Цель работы: Научиться работать с датами и временем в pandas, выполнять агрегирование по временным интервалам и использовать оконные функции для анализа активности пользователей и их сессий на основе логов заказов и событий.

## 1. Подготовка окружения и данных

### 1.1. Импорт библиотек

In [1]:
import pandas as pd
import numpy as np

### 1.2. Датасеты для работы

В работе используются уже подготовленные наборы данных из практики №3 (папка `shared/data`):

- Заказы интернет-магазина: `orders_2024_01.csv`, `orders_2024_02.csv`, `orders_2024_03.csv`.
- Логи событий/кликов: `events_records.json`, `events_lines.json`.

Структура заказов (CSV): `order_id`, `user_id`, `order_datetime` (строка), `product_id`, `price`, `quantity`, `status` (`paid`, `cancelled`, `refunded`).

Структура событий (JSON): `event_id`, `user_id`, `event_time`, `event_type` (`page_view`, `click`, `scroll`, `add_to_cart`, `purchase`), `page`.

## 2. Преобразование строк в datetime и работа с временными зонами

### 2.1. Загрузка заказов и событий

In [2]:
# Заказы
orders_files = [
    "../shared/data/orders_2024_01.csv",
    "../shared/data/orders_2024_02.csv",
    "../shared/data/orders_2024_03.csv",
]

df_orders = pd.concat(
    [pd.read_csv(f) for f in orders_files],
    ignore_index=True
)

# События (JSON records + JSON lines)
df_events_records = pd.read_json("../shared/data/events_records.json", orient="records")
df_events_lines = pd.read_json("../shared/data/events_lines.json", lines=True)

df_events = pd.concat(
    [df_events_records, df_events_lines],
    ignore_index=True
)

print(df_orders.head())
print(df_events.head())

   order_id  user_id       order_datetime  product_id     price  quantity  \
0         1     1044  2024-01-03 18:39:26          26  14038.15         1   
1         2     1079  2024-01-03 17:45:43          16   9172.60         1   
2         3     1041  2024-01-06 22:46:38          25   9035.26         2   
3         4     1083  2024-01-08 21:03:51           9  15234.33         3   
4         5     1068  2024-01-03 23:26:53          24   4134.38         2   

     status  
0  refunded  
1      paid  
2      paid  
3      paid  
4      paid  
   event_id  user_id          event_time   event_type         page
0         1     1095 2024-02-07 12:08:14  add_to_cart  /product/28
1         2     1058 2024-02-16 12:59:13     purchase   /product/7
2         3     1006 2024-01-19 00:14:21        click  /product/27
3         4     1092 2024-01-01 07:34:55       scroll  /product/26
4         5     1014 2024-02-17 19:46:47    page_view  /product/14


### 2.2. Парсинг `order_datetime` и `event_time` в datetime

In [3]:
df_orders["order_datetime"] = pd.to_datetime(df_orders["order_datetime"])
df_events["event_time"] = pd.to_datetime(df_events["event_time"])

print(df_orders["order_datetime"].dtype)
print(df_events["event_time"].dtype)

datetime64[us]
datetime64[us]


### 2.3. Выделение даты, часа и дня недели

In [4]:
df_orders["order_date"] = df_orders["order_datetime"].dt.date
df_orders["order_hour"] = df_orders["order_datetime"].dt.hour
df_orders["order_weekday"] = df_orders["order_datetime"].dt.day_name()

df_events["event_date"] = df_events["event_time"].dt.date
df_events["event_hour"] = df_events["event_time"].dt.hour
df_events["event_weekday"] = df_events["event_time"].dt.day_name()

print(df_orders[["order_datetime", "order_date", "order_hour", "order_weekday"]].head())
print(df_events[["event_time", "event_date", "event_hour", "event_weekday"]].head())

       order_datetime  order_date  order_hour order_weekday
0 2024-01-03 18:39:26  2024-01-03          18     Wednesday
1 2024-01-03 17:45:43  2024-01-03          17     Wednesday
2 2024-01-06 22:46:38  2024-01-06          22      Saturday
3 2024-01-08 21:03:51  2024-01-08          21        Monday
4 2024-01-03 23:26:53  2024-01-03          23     Wednesday
           event_time  event_date  event_hour event_weekday
0 2024-02-07 12:08:14  2024-02-07          12     Wednesday
1 2024-02-16 12:59:13  2024-02-16          12        Friday
2 2024-01-19 00:14:21  2024-01-19           0        Friday
3 2024-01-01 07:34:55  2024-01-01           7        Monday
4 2024-02-17 19:46:47  2024-02-17          19      Saturday


### 2.4. Временные зоны: назначение и смена

Предположим, что все времена в логах хранятся в UTC, а нам нужно перевести их в московское время (`Europe/Moscow`).

In [5]:
# Сначала локализуем как UTC
df_orders["order_datetime_utc"] = df_orders["order_datetime"].dt.tz_localize("UTC")
df_events["event_time_utc"] = df_events["event_time"].dt.tz_localize("UTC")

# Переводим в часовую зону Москвы
df_orders["order_datetime_msk"] = df_orders["order_datetime_utc"].dt.tz_convert("Europe/Moscow")
df_events["event_time_msk"] = df_events["event_time_utc"].dt.tz_convert("Europe/Moscow")

print(df_orders[["order_datetime", "order_datetime_utc", "order_datetime_msk"]].head())
print(df_events[["event_time", "event_time_utc", "event_time_msk"]].head())

       order_datetime        order_datetime_utc        order_datetime_msk
0 2024-01-03 18:39:26 2024-01-03 18:39:26+00:00 2024-01-03 21:39:26+03:00
1 2024-01-03 17:45:43 2024-01-03 17:45:43+00:00 2024-01-03 20:45:43+03:00
2 2024-01-06 22:46:38 2024-01-06 22:46:38+00:00 2024-01-07 01:46:38+03:00
3 2024-01-08 21:03:51 2024-01-08 21:03:51+00:00 2024-01-09 00:03:51+03:00
4 2024-01-03 23:26:53 2024-01-03 23:26:53+00:00 2024-01-04 02:26:53+03:00
           event_time            event_time_utc            event_time_msk
0 2024-02-07 12:08:14 2024-02-07 12:08:14+00:00 2024-02-07 15:08:14+03:00
1 2024-02-16 12:59:13 2024-02-16 12:59:13+00:00 2024-02-16 15:59:13+03:00
2 2024-01-19 00:14:21 2024-01-19 00:14:21+00:00 2024-01-19 03:14:21+03:00
3 2024-01-01 07:34:55 2024-01-01 07:34:55+00:00 2024-01-01 10:34:55+03:00
4 2024-02-17 19:46:47 2024-02-17 19:46:47+00:00 2024-02-17 22:46:47+03:00


## 3. Агрегация по времени: resample для заказов и событий

### 3.1. Подготовка индекса по времени

Для ресемплинга удобно сделать временной столбец индексом. Перед этим вычислим выручку по строке заказа.

In [6]:
# Копии с временным индексом
orders_ts = df_orders.set_index("order_datetime_msk").sort_index()
events_ts = df_events.set_index("event_time_msk").sort_index()

orders_ts["revenue"] = orders_ts["price"] * orders_ts["quantity"]

### 3.2. Агрегация заказов по дням и часам

В pandas 3 часовая частота обозначается строчной буквой `"h"` (вариант `"H"` из методички удалён).

In [7]:
# По дням
daily_orders = orders_ts.resample("D").agg(
    orders_count=("order_id", "nunique"),
    revenue_sum=("revenue", "sum")
)
print(daily_orders.head())

# По часам
hourly_orders = orders_ts.resample("h").agg(
    orders_count=("order_id", "nunique"),
    revenue_sum=("revenue", "sum")
)
print(hourly_orders.head())

                           orders_count  revenue_sum
order_datetime_msk                                  
2024-01-01 00:00:00+03:00           141   3888229.53
2024-01-02 00:00:00+03:00           165   4242645.43
2024-01-03 00:00:00+03:00           169   4517004.59
2024-01-04 00:00:00+03:00           168   3624982.14
2024-01-05 00:00:00+03:00           181   4763831.59
                           orders_count  revenue_sum
order_datetime_msk                                  
2024-01-01 03:00:00+03:00             6    179435.77
2024-01-01 04:00:00+03:00             6    154360.91
2024-01-01 05:00:00+03:00             3     86862.23
2024-01-01 06:00:00+03:00             3    146437.87
2024-01-01 07:00:00+03:00             6    176851.24


### 3.3. Агрегация событий по типам и дням

In [8]:
# Количество событий по типам за день
daily_events = events_ts.resample("D")["event_type"].value_counts().unstack(fill_value=0)
print(daily_events.head())

event_type                 add_to_cart  click  page_view  purchase  scroll
event_time_msk                                                            
2024-01-01 00:00:00+03:00            2      4          9        10       7
2024-01-02 00:00:00+03:00            8      6          9         6       4
2024-01-03 00:00:00+03:00            5      5          4        10      10
2024-01-04 00:00:00+03:00            6      5          9         6       3
2024-01-05 00:00:00+03:00            6      9          9         8       8


### 3.4. Агрегация по неделям

In [9]:
weekly_orders = orders_ts.resample("W").agg(
    orders_count=("order_id", "nunique"),
    revenue_sum=("revenue", "sum")
)

weekly_events = events_ts.resample("W")["event_id"].count().to_frame("events_count")

print(weekly_orders.head())
print(weekly_events.head())

                           orders_count  revenue_sum
order_datetime_msk                                  
2024-01-07 00:00:00+03:00          1142  29454307.30
2024-01-14 00:00:00+03:00          1150  30518882.26
2024-01-21 00:00:00+03:00          1137  29762097.01
2024-01-28 00:00:00+03:00          1117  28413211.87
2024-02-04 00:00:00+03:00          1131  28362191.53
                           events_count
event_time_msk                         
2024-01-07 00:00:00+03:00           242
2024-01-14 00:00:00+03:00           211
2024-01-21 00:00:00+03:00           244
2024-01-28 00:00:00+03:00           249
2024-02-04 00:00:00+03:00           346


## 4. Rolling и expanding окна для скользящих метрик

### 4.1. Скользящее среднее выручки по дням

In [10]:
# Используем daily_orders из предыдущего раздела
daily_orders["revenue_rolling_7d"] = (
    daily_orders["revenue_sum"]
    .rolling(window=7, min_periods=1)
    .mean()
)

print(daily_orders[["revenue_sum", "revenue_rolling_7d"]].head(15))

                           revenue_sum  revenue_rolling_7d
order_datetime_msk                                        
2024-01-01 00:00:00+03:00   3888229.53        3.888230e+06
2024-01-02 00:00:00+03:00   4242645.43        4.065437e+06
2024-01-03 00:00:00+03:00   4517004.59        4.215960e+06
2024-01-04 00:00:00+03:00   3624982.14        4.068215e+06
2024-01-05 00:00:00+03:00   4763831.59        4.207339e+06
2024-01-06 00:00:00+03:00   4450229.73        4.247821e+06
2024-01-07 00:00:00+03:00   3967384.29        4.207758e+06
2024-01-08 00:00:00+03:00   4201864.84        4.252563e+06
2024-01-09 00:00:00+03:00   4442862.12        4.281166e+06
2024-01-10 00:00:00+03:00   4249550.74        4.242958e+06
2024-01-11 00:00:00+03:00   4415233.33        4.355851e+06
2024-01-12 00:00:00+03:00   3630431.00        4.193937e+06
2024-01-13 00:00:00+03:00   4310497.15        4.173975e+06
2024-01-14 00:00:00+03:00   5268443.08        4.359840e+06
2024-01-15 00:00:00+03:00   4395603.31        4.387517e+

### 4.2. Накопительная (expanding) сумма выручки

In [11]:
daily_orders["revenue_expanding"] = daily_orders["revenue_sum"].expanding().sum()

print(daily_orders[["revenue_sum", "revenue_expanding"]].head(15))

                           revenue_sum  revenue_expanding
order_datetime_msk                                       
2024-01-01 00:00:00+03:00   3888229.53         3888229.53
2024-01-02 00:00:00+03:00   4242645.43         8130874.96
2024-01-03 00:00:00+03:00   4517004.59        12647879.55
2024-01-04 00:00:00+03:00   3624982.14        16272861.69
2024-01-05 00:00:00+03:00   4763831.59        21036693.28
2024-01-06 00:00:00+03:00   4450229.73        25486923.01
2024-01-07 00:00:00+03:00   3967384.29        29454307.30
2024-01-08 00:00:00+03:00   4201864.84        33656172.14
2024-01-09 00:00:00+03:00   4442862.12        38099034.26
2024-01-10 00:00:00+03:00   4249550.74        42348585.00
2024-01-11 00:00:00+03:00   4415233.33        46763818.33
2024-01-12 00:00:00+03:00   3630431.00        50394249.33
2024-01-13 00:00:00+03:00   4310497.15        54704746.48
2024-01-14 00:00:00+03:00   5268443.08        59973189.56
2024-01-15 00:00:00+03:00   4395603.31        64368792.87


### 4.3. Скользящее количество событий

In [12]:
# Пример по суммарному числу событий в день
events_daily_count = events_ts.resample("D")["event_id"].count().to_frame("events_count")

events_daily_count["events_rolling_3d"] = (
    events_daily_count["events_count"]
    .rolling(window=3, min_periods=1)
    .sum()
)

print(events_daily_count.head(10))

                           events_count  events_rolling_3d
event_time_msk                                            
2024-01-01 00:00:00+03:00            32               32.0
2024-01-02 00:00:00+03:00            33               65.0
2024-01-03 00:00:00+03:00            34               99.0
2024-01-04 00:00:00+03:00            29               96.0
2024-01-05 00:00:00+03:00            40              103.0
2024-01-06 00:00:00+03:00            42              111.0
2024-01-07 00:00:00+03:00            32              114.0
2024-01-08 00:00:00+03:00            27              101.0
2024-01-09 00:00:00+03:00            34               93.0
2024-01-10 00:00:00+03:00            28               89.0


## 5. Оконные функции по пользователю и анализ сессий

### 5.1. Подготовка данных событий по пользователям

In [13]:
# Сортируем события по пользователю и времени
df_events_user = (
    df_events
    .sort_values(["user_id", "event_time_msk"])
    .copy()
)

### 5.2. Время между событиями (межсобытийный интервал)

In [14]:
df_events_user["prev_event_time"] = (
    df_events_user
    .groupby("user_id")["event_time_msk"]
    .shift(1)
)

df_events_user["delta_since_prev"] = (
    df_events_user["event_time_msk"] - df_events_user["prev_event_time"]
)

print(df_events_user[["user_id", "event_time_msk", "prev_event_time", "delta_since_prev"]].head(10))

      user_id            event_time_msk           prev_event_time  \
930      1001 2024-01-03 08:57:47+03:00                       NaT   
1591     1001 2024-01-04 10:42:32+03:00 2024-01-03 08:57:47+03:00   
599      1001 2024-01-06 05:06:36+03:00 2024-01-04 10:42:32+03:00   
1215     1001 2024-01-07 03:43:21+03:00 2024-01-06 05:06:36+03:00   
676      1001 2024-01-07 10:17:51+03:00 2024-01-07 03:43:21+03:00   
893      1001 2024-01-12 21:36:41+03:00 2024-01-07 10:17:51+03:00   
1411     1001 2024-01-24 05:04:00+03:00 2024-01-12 21:36:41+03:00   
1644     1001 2024-01-25 03:41:23+03:00 2024-01-24 05:04:00+03:00   
1152     1001 2024-01-26 13:25:08+03:00 2024-01-25 03:41:23+03:00   
1413     1001 2024-01-27 21:28:01+03:00 2024-01-26 13:25:08+03:00   

     delta_since_prev  
930               NaT  
1591  1 days 01:44:45  
599   1 days 18:24:04  
1215  0 days 22:36:45  
676   0 days 06:34:30  
893   5 days 11:18:50  
1411 11 days 07:27:19  
1644  0 days 22:37:23  
1152  1 days 09:43:45  


### 5.3. Определение сессий по таймауту

Принято считать, что новая сессия начинается, если перерыв между событиями пользователя больше, например, 30 минут.

In [15]:
session_timeout = pd.Timedelta(minutes=30)

# Флаг начала новой сессии
df_events_user["is_new_session"] = (
    df_events_user["delta_since_prev"].isna() |
    (df_events_user["delta_since_prev"] > session_timeout)
)

# Нумеруем сессии внутри каждого пользователя
df_events_user["session_id"] = (
    df_events_user.groupby("user_id")["is_new_session"].cumsum()
)

print(df_events_user[["user_id", "event_time_msk", "delta_since_prev", "is_new_session", "session_id"]].head(10))

      user_id            event_time_msk delta_since_prev  is_new_session  \
930      1001 2024-01-03 08:57:47+03:00              NaT            True   
1591     1001 2024-01-04 10:42:32+03:00  1 days 01:44:45            True   
599      1001 2024-01-06 05:06:36+03:00  1 days 18:24:04            True   
1215     1001 2024-01-07 03:43:21+03:00  0 days 22:36:45            True   
676      1001 2024-01-07 10:17:51+03:00  0 days 06:34:30            True   
893      1001 2024-01-12 21:36:41+03:00  5 days 11:18:50            True   
1411     1001 2024-01-24 05:04:00+03:00 11 days 07:27:19            True   
1644     1001 2024-01-25 03:41:23+03:00  0 days 22:37:23            True   
1152     1001 2024-01-26 13:25:08+03:00  1 days 09:43:45            True   
1413     1001 2024-01-27 21:28:01+03:00  1 days 08:02:53            True   

      session_id  
930            1  
1591           2  
599            3  
1215           4  
676            5  
893            6  
1411           7  
1644       

### 5.4. Длина сессий и количество событий в сессии

In [16]:
session_stats = (
    df_events_user
    .groupby(["user_id", "session_id"])
    .agg(
        session_start=("event_time_msk", "min"),
        session_end=("event_time_msk", "max"),
        events_in_session=("event_id", "count")
    )
    .reset_index()
)

session_stats["session_duration"] = session_stats["session_end"] - session_stats["session_start"]

print(session_stats.head())

   user_id  session_id             session_start               session_end  \
0     1001           1 2024-01-03 08:57:47+03:00 2024-01-03 08:57:47+03:00   
1     1001           2 2024-01-04 10:42:32+03:00 2024-01-04 10:42:32+03:00   
2     1001           3 2024-01-06 05:06:36+03:00 2024-01-06 05:06:36+03:00   
3     1001           4 2024-01-07 03:43:21+03:00 2024-01-07 03:43:21+03:00   
4     1001           5 2024-01-07 10:17:51+03:00 2024-01-07 10:17:51+03:00   

   events_in_session session_duration  
0                  1           0 days  
1                  1           0 days  
2                  1           0 days  
3                  1           0 days  
4                  1           0 days  


### 5.5. Ранжирование событий пользователя (оконная функция rank)

In [17]:
df_events_user["event_rank_in_session"] = (
    df_events_user
    .groupby(["user_id", "session_id"])["event_time_msk"]
    .rank(method="first")
)

print(df_events_user[["user_id", "session_id", "event_time_msk", "event_rank_in_session"]].head(10))

      user_id  session_id            event_time_msk  event_rank_in_session
930      1001           1 2024-01-03 08:57:47+03:00                    1.0
1591     1001           2 2024-01-04 10:42:32+03:00                    1.0
599      1001           3 2024-01-06 05:06:36+03:00                    1.0
1215     1001           4 2024-01-07 03:43:21+03:00                    1.0
676      1001           5 2024-01-07 10:17:51+03:00                    1.0
893      1001           6 2024-01-12 21:36:41+03:00                    1.0
1411     1001           7 2024-01-24 05:04:00+03:00                    1.0
1644     1001           8 2024-01-25 03:41:23+03:00                    1.0
1152     1001           9 2024-01-26 13:25:08+03:00                    1.0
1413     1001          10 2024-01-27 21:28:01+03:00                    1.0
